# GP Metadata phenotypes

### Spark and Hail setup

In [ ]:
import pyspark
import dxpy
import hail as hl

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)
hl.init(sc=sc, default_reference='GRCh38')

### Environment setup check

In [ ]:
from datetime import datetime
print(f'Timestamp: {datetime.now()}')
print(f'Project: {dxpy.describe(dxpy.PROJECT_CONTEXT_ID)["name"]} ({dxpy.PROJECT_CONTEXT_ID})')
print(f'Instance type: {dxpy.describe(dxpy.JOB_ID)["instanceType"]}')
print(f'Hail version: {hl.version()}')
print(f'Spark version: {spark.version}')

### Importing libraries

In [ ]:
import os

from pyspark.sql.functions import col, least, greatest, datediff, to_date
from pyspark.sql.types import DateType

### Parameters and configuration (hail, spark)

In [ ]:
dispensed_database_name = 'app879030_20250811132217'

prescription_db_name = 'prescriptions_db'
output_phenotypes_tb = 'gp_metadata.ht'

hl_preffered_partitioning = 24

primary_provider_max_ratio = 0.8

In [ ]:
prescriptions_db_id = dxpy.find_one_data_object(name=prescription_db_name, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']
output_phenotypes_tb_url = f"dnax://{prescriptions_db_id}/{output_phenotypes_tb}"

In [ ]:
spark.sql(f'USE {dispensed_database_name}')

In [ ]:
providers_map = {
    'england_vision': 1,
    'scotland': 2,
    'england_tpp': 3,
    'wales': 4
}

## Extracting data providers phenotypes from prescriptions (GP) database

In [ ]:
%%time
gp_scripts_tb = spark.sql('SELECT eid, data_provider, issue_date AS timestamp FROM gp_scripts')
gp_scripts_tb = gp_scripts_tb.withColumn('timestamp', gp_scripts_tb.timestamp.cast('timestamp').cast('int'))
data_providers = hl.Table.from_spark(gp_scripts_tb).cache()
data_providers.count()

In [ ]:
provider_counts = data_providers.group_by('eid', 'data_provider').aggregate(provider_count = hl.agg.count()).cache()

In [ ]:
spark_cnt = spark.sql("SELECT COUNT(1) FROM (SELECT DISTINCT eid, data_provider FROM gp_scripts)").first()[0]
assert provider_counts.count() == spark_cnt

In [ ]:
gp_scripts_eids_count = spark.sql("SELECT COUNT(1) FROM (SELECT DISTINCT eid FROM gp_scripts)").first()[0]

#### Latest provider by prescription date

In [ ]:
last_providers = data_providers.group_by('eid').aggregate(
    last_provider = hl.agg.take(data_providers.row, 1, ordering=-data_providers.timestamp)[0].data_provider
).key_by('eid').cache()

In [ ]:
assert last_providers.count() == gp_scripts_eids_count

#### Primary provider by number of prescriptions

In [ ]:
primary_providers = provider_counts.group_by('eid').aggregate(
    providers = hl.sorted(hl.agg.collect(provider_counts.row), lambda row: row.provider_count, reverse=True)
).cache()

In [ ]:
print(f'providers == 1: {primary_providers.filter(hl.len(primary_providers.providers) == 1).count()}')
print(f'providers > 1: {primary_providers.filter(hl.len(primary_providers.providers) > 1).count()}')
print(f'providers > 2: {primary_providers.filter(hl.len(primary_providers.providers) > 2).count()}')

In [ ]:
primary_providers = primary_providers.annotate(
    primary_provider = primary_providers.providers[0].data_provider,
    ratio = hl.if_else(
        hl.len(primary_providers.providers) > 1,
        primary_providers.providers[1].provider_count / primary_providers.providers[0].provider_count,
        hl.missing(hl.tfloat)
    )
).cache()

In [ ]:
primary_providers = primary_providers.key_by('eid').cache()

In [ ]:
assert primary_providers.count() == gp_scripts_eids_count

#### Picking primary or latest provider based on provider prescriptions ratio

In [ ]:
primary_providers = primary_providers.annotate(last_provider = last_providers[primary_providers.eid].last_provider)
primary_providers = primary_providers.select('primary_provider').cache()

In [ ]:
hl_provides_list = hl.literal([str(i) for i in list(providers_map.values())])
assert primary_providers.filter(~hl_provides_list.contains(primary_providers.primary_provider)).count() == 0
primary_providers = primary_providers.annotate(primary_provider = hl.int(primary_providers.primary_provider)).cache()

#### Building phenotype

In [ ]:
annotations = {}
for provider_name, provider_id in providers_map.items():
    annotations[f'prescription_data_provider__{provider_name}'] = (primary_providers.primary_provider == provider_id)

In [ ]:
%%time
primary_providers = primary_providers.annotate(**annotations).repartition(hl_preffered_partitioning).cache()

## Extracting age first seen and age last seen from GP records
- gp_clinical
- gp_scripts

### Selecting dates of first events and issues

In [ ]:
gp_clinical_dates = spark.sql("""
    SELECT eid,
           MIN(event_dt) as first_gp_clinical,
           MAX(event_dt) as last_gp_clinical
    FROM gp_clinical
    WHERE (event_dt >= '1945-01-01'
    AND event_dt <= '2025-01-01')
    GROUP BY eid
    ORDER BY eid
""")

gp_scripts_dates = spark.sql("""
    SELECT eid,
           MIN(issue_date) as first_gp_prescription,
           MAX(issue_date) as last_gp_prescription
    FROM gp_scripts
    WHERE (issue_date >= '1945-01-01'
    AND issue_date <= '2025-01-01')
    GROUP BY eid
    ORDER BY eid
""")

In [ ]:
dates = gp_clinical_dates.join(gp_scripts_dates, on="eid", how="inner")

In [ ]:
dates = dates.withColumn("first_seen_gp", least(col("first_gp_clinical"), col("first_gp_prescription")))
dates = dates.withColumn("last_seen_gp", greatest(col("last_gp_clinical"), col("last_gp_prescription")))

dates = dates.withColumn("prescription_records_coverage", datediff(col("last_gp_prescription"), col("first_gp_prescription")))
dates = dates.withColumn("clinical_records_coverage",datediff(col("last_gp_clinical"), col("first_gp_clinical")))
dates = dates.withColumn("gp_records_coverage", datediff(col("last_seen_gp"), col("first_seen_gp")))

In [ ]:
date_columns = [field.name for field in dates.schema.fields if isinstance(field.dataType, DateType)]
for i in date_columns:
    dates = dates.withColumn(i, col(i).cast("string"))

### Export to hail

In [ ]:
dates_hail = hl.Table.from_spark(dates)

In [ ]:
dates_hail = dates_hail.key_by("eid").cache()

In [ ]:
date_format = "%Y-%m-%d %H:%M:%S"

dates_hail  = dates_hail.transmute(
    **{
        col: hl.experimental.strptime(dates_hail[col] + ' 00:00:00', date_format,'GMT')
        for col in date_columns
    }
)

dates_hail = dates_hail.cache()

### Checking data integrity

In [ ]:
gp_clinical_eids = hl.Table.from_spark(spark.sql('SELECT DISTINCT eid from gp_clinical')).key_by('eid').cache()
gp_scripts_eids = hl.Table.from_spark(spark.sql('SELECT DISTINCT eid from gp_scripts')).key_by('eid').cache()

In [ ]:
print(f'dates_hail: {dates_hail.count()}')
print(f'primary_providers: {primary_providers.count()}')
print(f'gp_clinical: {gp_clinical_eids.count()}')
print(f'gp_scripts: {gp_scripts_eids.count()}')

In [ ]:
gp_clinical_eids.anti_join(dates_hail).count()

In [ ]:
gp_scripts_eids.anti_join(dates_hail).count()

In [ ]:
primary_providers.anti_join(dates_hail).count()

### Annotating result to the table with prescriptions data providers

In [ ]:
gp_metadata = primary_providers.annotate(
    **dates_hail[primary_providers.eid]
).drop("primary_provider")

In [ ]:
gp_metadata = gp_metadata.repartition(hl_preffered_partitioning).cache()

In [ ]:
gp_metadata.describe()

## Write result Hail table to dnax database

In [ ]:
%time gp_metadata.write(output_phenotypes_tb_url, overwrite = True)

## Visualization

In [ ]:
df = gp_metadata.to_pandas()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt



# --- identyfikacja typów zmiennych ---
binary_vars = []
continuous_vars = []

for col in df.columns:
    if df[col].dropna().isin([0, 1]).all():
        binary_vars.append(col)
    elif pd.api.types.is_numeric_dtype(df[col]):
        continuous_vars.append(col)

print("Binary variables:", binary_vars)
print("Continuous variables:", continuous_vars)

# --- konwersja kolumn z timestampów sekundowych ---
for col in df.columns:
    if any(keyword in col.lower() for keyword in ["first", "last", "date", "seen", "time"]):
        df[col] = pd.to_datetime(df[col], unit="s", errors="coerce")



# --- wykresy binarne ---
if binary_vars:
    bin_counts = df[binary_vars].sum().sort_values(ascending=False)
    plt.figure(figsize=(10, 5))
    bin_counts.plot(kind="bar")
    plt.title("Binary variables (count of 1s)")
    plt.ylabel("Count of 1")
    plt.savefig("Prescriptions data providers participants count")
    plt.show()

# --- histogramy zmiennych ciągłych ---
for col in continuous_vars:
    plt.figure(figsize=(7, 4))
    df[col].dropna().hist(bins=30)
    plt.title(f"Histogram of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.savefig(f"Histogram of {col}")
    plt.show()

# --- histogramy kolumn datowych ---
date_cols = [col for col in df.columns if pd.api.types.is_datetime64_any_dtype(df[col])]

for col in date_cols:
    plt.figure(figsize=(8, 4))
    df[col].dropna().hist(bins=50)
    plt.title(f"Histogram of {col} (dates)")
    plt.xlabel("Date")
    plt.ylabel("Count")
    plt.savefig(f"Histogram of {col} (dates)")
    plt.show()


In [ ]:
# --- SCATTERPLOTY: first_* vs last_* ---
first_cols = [c for c in date_cols if "first" in c.lower()]
last_cols = [c for c in date_cols if "last" in c.lower()]

# mapowanie par
pairs = []
for f in first_cols:
    suffix = f.lower().replace("first", "")
    match = [l for l in last_cols if suffix in l.lower()]
    if match:
        pairs.append((f, match[0]))

print("\nPary first/last znalezione:")
for f, l in pairs:
    print(f"  {f} ↔ {l}")
    
    
# --- przygotowanie zmiennej 'provider' ---
provider_cols = [
    'prescription_data_provider__england_vision',
    'prescription_data_provider__scotland',
    'prescription_data_provider__england_tpp',
    'prescription_data_provider__wales'
]


def get_provider(row):
    active = [p.replace('prescription_data_provider__', '').replace('_', ' ').title() 
              for p in provider_cols if row.get(p, 0) == 1]
    if len(active) == 1:
        return active[0]
    elif len(active) > 1:
        return "Multiple"
    else:
        return "None"

df["provider_label"] = df[provider_cols].apply(get_provider, axis=1)

In [ ]:
# --- mapa kolorów ---
color_map = {
    "England Tpp": "#ff7f0e",
    "Scotland": "#2ca02c",
    "Wales": "#9467bd",
    "England Vision": "#1f77b4",
    "Multiple": "#d62728",
    "None": "#7f7f7f"
}


# --- rysowanie scatterplotów z kolorami ---
for f, l in pairs:
    plt.figure(figsize=(6, 6))
    for provider, color in color_map.items():
        subset = df[df["provider_label"] == provider]
        plt.scatter(subset[f], subset[l], s=5, alpha=0.6, label=provider, color=color)
    
    # linia przekątna (idealnie first = last)
    min_date = min(df[f].min(), df[l].min())
    max_date = max(df[f].max(), df[l].max())
    plt.plot([min_date, max_date], [min_date, max_date], color="black", linestyle="--", lw=1,alpha=0.01)
    plt.xlabel(f)
    plt.ylabel(l)
    plt.title(f"{f} vs {l} by Provider")
    plt.legend(markerscale=4, fontsize="small", frameon=False)
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{f} vs {l} by Provider.png")
    plt.show()